In [2]:
%pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [3]:
from pypdf import PdfReader

reader = PdfReader("/home/linux00/Downloads/p/Resumes/GVyshnavi_.pdf")

content = ""

for page in reader.pages:
    text = page.extract_text()
    if text:
        content += text + "\n"

#print(content)

In [4]:
%pip install tiktoken

Note: you may need to restart the kernel to use updated packages.


In [5]:
import tiktoken

# Choose tokenizer
encoding = tiktoken.get_encoding("cl100k_base")

def token_chunk(text, chunk_size=100, overlap=10):
    """
    Split text into overlapping token chunks.

    Args:
        text: Input text
        chunk_size: Number of tokens per chunk
        overlap: Number of overlapping tokens

    Returns:
        List of text chunks
    """

    tokens = encoding.encode(text)

    chunks = []
    start = 0

    while start < len(tokens):
        end = start + chunk_size

        chunk_tokens = tokens[start:end]
        chunk_text = encoding.decode(chunk_tokens)

        chunks.append(chunk_text)

        start += chunk_size - overlap

    return chunks

In [6]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

def get_embedding(text):
    embedding = embedder.encode(text)

    #print(len(embedding))
    return embedding

/home/linux00/.local/lib/python3.8/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [7]:
chunks = token_chunk(content)

vector_store = []

for idx, chunk in enumerate(chunks):
    #print(chunk)
    embedding = get_embedding(chunk)
    #print(embedding)
    vector_store.append({
        "id": idx,
        "text": chunk,
        "embedding": embedding
    })

print(f"Stored {len(vector_store)} chunks")

Stored 9 chunks


In [8]:
import numpy as np

def cosine_similarity(v1, v2):
    v1 = np.array(v1)
    v2 = np.array(v2)

    return np.dot(v1, v2) / (
        np.linalg.norm(v1) * np.linalg.norm(v2)
    )

In [9]:
def retrieve(query, vector_store, top_k=3):

    query_embedding = get_embedding(query)

    scores = []

    for record in vector_store:

        score = cosine_similarity(
            query_embedding,
            record["embedding"]
        )

        scores.append(
            (score, record)
        )

    scores.sort(reverse=True, key=lambda x: x[0])

    return scores[:top_k]

In [10]:
query = "How many years of experience"

results = retrieve(query, vector_store)

for score, chunk in results:

    print("\n----------------")
    print("Score:", round(score, 4))
    print(chunk["text"])


----------------
Score: 0.2592
Full-time) March 2023 - Aug 2023
◦ Contributed to the full software development lifecycle of IEAC, an application that digitized audit processes through
automated data collection.
◦ Translated business requirements into robust data models for efficient data management.
◦ Created an Audit Calculation Component.
◦ Implemented features resulting in a 30% reduction in audit process time.
◦ Technologies: Java, Spring MVC, Hibernate, REST, MySQL, ReactJS
• Accent

----------------
Score: 0.1936
ered a navigation service utilizing geospatial data.
◦ Developed and implemented algorithms for computing Out-of-Home (OOH) industry metrics.
◦ Constructed reusable Java libraries to enhance code maintainability and streamline development processes.
◦ Technologies: Java 17, JWT, Spring MVC, Microservices, REST, MongoDB, JSP, MySQL, Multithreading, OSM
• AdMAVIN Chennai
IEAC — Software Developer (Full-time) March 2023 - Aug 

----------------
Score: 0.1728
 tracking Indi